# Case Study 13: Breast Cancer Outcomes — Binomial GAMM (PQL)
## Aurora-GLM Showcase: Logistic Mixed Model for Multi-Center Medical Data

---

## 1. Overview

This notebook demonstrates a **binomial GAMM** (logistic mixed model fitted by Penalized Quasi-Likelihood) for a multi-center breast cancer study. The outcome is **5-year survival (binary: survived / not survived)** and a random intercept per hospital accounts for center-level variation.

> **What this is NOT.** Despite the historical title of this dataset, this is **not a survival analysis** in the technical sense: there is no censoring, no time-to-event variable, no hazard function, and no Kaplan–Meier or Cox model. We model a binary endpoint (alive at 5 years, yes/no). For true time-to-event data you need a survival framework, which Aurora-GLM does not currently provide.

### Research Questions

1. Which patient factors predict 5-year survival?
2. Is the age effect non-linear (quadratic)?
3. How much variation exists between hospitals (ICC)?
4. How do estimates change when clustering is ignored (GLM vs GAMM)?

### Aurora-GLM Capabilities

1. Binomial GAMM via PQL (`fit_gamm(..., family='binomial')`)
2. Random intercepts for hospital clustering
3. ICC calculation (latent-variable scale)
4. Odds-ratio interpretation
5. Conditional (GAMM) vs marginal (GLM) effect comparison

---


## 2. Setup and Data Generation

Simulated multi-center study: 30 hospitals × 50 patients (N=1,500, seed 42). The data-generating process — including the hospital random-effect strength (sd = 1.0 on the logit scale, chosen so PQL can recover it) — is documented in the generation cell.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import time

import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from aurora.models.gamm import fit_gamm
from aurora.models.glm import fit_glm

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_context('notebook', font_scale=1.1)
%config InlineBackend.figure_format = 'retina'
np.random.seed(42)

print("="*80)
print("ENVIRONMENT SETUP")
print("="*80)
print("Backend: NumPy (PQL estimation path)")
print("="*80)


In [ ]:
# Generate synthetic breast cancer survival data
# (Simulating multi-center study structure)
print("="*80)
print("DATA GENERATION")
print("="*80)

np.random.seed(42)

# Parameters
n_hospitals = 30
n_patients_per_hospital = 50
n_total = n_hospitals * n_patients_per_hospital

# Hospital random effects.
# NOTE: sd = 1.0 on the logit scale (a sizable but realistic center effect,
# ICC ~ 0.2). An earlier version used sd = 0.5, but the PQL estimator cannot
# recover such a weak variance component from binary data and collapses it
# to zero (see the Limitations section) - the stronger signal makes the
# clustering recoverable and the teaching points honest.
hospital_effects = np.random.normal(0, 1.0, n_hospitals)

# Patient data
hospital_id = np.repeat(np.arange(n_hospitals), n_patients_per_hospital)
age = np.random.normal(55, 12, n_total).clip(25, 85)
tumor_size = np.random.lognormal(2.5, 0.6, n_total).clip(5, 100)  # mm
grade = np.random.choice([1, 2, 3], n_total, p=[0.2, 0.5, 0.3])
er_positive = np.random.binomial(1, 0.7, n_total)
lymph_nodes = np.random.poisson(1.5, n_total).clip(0, 15)
chemo = np.random.binomial(1, 0.6, n_total)

# Generate survival outcome (5-year survival)
# Logistic model with non-linear age effect
age_effect = -0.02 * (age - 50) + 0.001 * (age - 50)**2  # U-shaped
tumor_effect = -0.03 * tumor_size
grade_effect = -0.4 * (grade - 1)
er_effect = 0.5 * er_positive
ln_effect = -0.2 * lymph_nodes
chemo_effect = 0.3 * chemo

linear_pred = (2 + age_effect + tumor_effect + grade_effect + 
               er_effect + ln_effect + chemo_effect + 
               hospital_effects[hospital_id])

prob_survival = 1 / (1 + np.exp(-linear_pred))
survived = np.random.binomial(1, prob_survival)

# Create DataFrame
df = pd.DataFrame({
    'hospital_id': hospital_id,
    'age': age,
    'tumor_size': tumor_size,
    'grade': grade,
    'er_positive': er_positive,
    'lymph_nodes': lymph_nodes,
    'chemo': chemo,
    'survived': survived
})

# Standardize continuous variables
df['age_std'] = (df['age'] - df['age'].mean()) / df['age'].std()
df['tumor_std'] = (df['tumor_size'] - df['tumor_size'].mean()) / df['tumor_size'].std()
df['ln_std'] = (df['lymph_nodes'] - df['lymph_nodes'].mean()) / df['lymph_nodes'].std()
df['age_std_sq'] = df['age_std'] ** 2  # quadratic age term (U-shaped effect in the DGP)

print(f"\nDataset Summary:")
print(f"   Patients: {n_total}")
print(f"   Hospitals: {n_hospitals}")
print(f"   5-year survival rate: {df['survived'].mean()*100:.1f}%")
print(f"\nVariable Ranges:")
print(f"   Age: {df['age'].min():.0f} - {df['age'].max():.0f} years")
print(f"   Tumor size: {df['tumor_size'].min():.0f} - {df['tumor_size'].max():.0f} mm")
print(f"   Lymph nodes: {df['lymph_nodes'].min()} - {df['lymph_nodes'].max()}")

print("="*80)

## 3. Exploratory Data Analysis

In [ ]:
# EDA
print("="*80)
print("EXPLORATORY DATA ANALYSIS")
print("="*80)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Survival by hospital
hospital_survival = df.groupby('hospital_id')['survived'].mean()
axes[0, 0].hist(hospital_survival, bins=15, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(hospital_survival.mean(), color='red', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Survival Rate')
axes[0, 0].set_ylabel('Number of Hospitals')
axes[0, 0].set_title('Hospital-Level Survival Rates', fontweight='bold')

# Age vs survival
age_bins = pd.cut(df['age'], bins=10)
age_survival = df.groupby(age_bins, observed=True)['survived'].mean()
centers = [i.mid for i in age_survival.index]
axes[0, 1].plot(centers, age_survival.values, 'o-', color='coral', linewidth=2)
axes[0, 1].set_xlabel('Age')
axes[0, 1].set_ylabel('Survival Rate')
axes[0, 1].set_title('Survival by Age (Non-linear!)', fontweight='bold')

# Tumor size vs survival
tumor_bins = pd.cut(df['tumor_size'], bins=10)
tumor_survival = df.groupby(tumor_bins, observed=True)['survived'].mean()
centers = [i.mid for i in tumor_survival.index]
axes[0, 2].plot(centers, tumor_survival.values, 'o-', color='seagreen', linewidth=2)
axes[0, 2].set_xlabel('Tumor Size (mm)')
axes[0, 2].set_ylabel('Survival Rate')
axes[0, 2].set_title('Survival by Tumor Size', fontweight='bold')

# Grade
grade_survival = df.groupby('grade')['survived'].mean()
axes[1, 0].bar(grade_survival.index, grade_survival.values, color='purple', alpha=0.7)
axes[1, 0].set_xlabel('Tumor Grade')
axes[1, 0].set_ylabel('Survival Rate')
axes[1, 0].set_title('Survival by Grade', fontweight='bold')

# ER status
er_survival = df.groupby('er_positive')['survived'].mean()
axes[1, 1].bar(['ER-', 'ER+'], er_survival.values, color=['salmon', 'lightgreen'])
axes[1, 1].set_ylabel('Survival Rate')
axes[1, 1].set_title('Survival by ER Status', fontweight='bold')

# Chemotherapy
chemo_survival = df.groupby('chemo')['survived'].mean()
axes[1, 2].bar(['No Chemo', 'Chemo'], chemo_survival.values, color=['lightblue', 'orange'])
axes[1, 2].set_ylabel('Survival Rate')
axes[1, 2].set_title('Survival by Chemotherapy', fontweight='bold')

plt.tight_layout()
plt.show()
print("="*80)

## 4. Mathematical Specification

### Binomial GAMM (Logistic Mixed Model)

For patient $i$ in hospital $j$:

$$Y_{ij} \sim \text{Bernoulli}(\pi_{ij})$$

$$\text{logit}(\pi_{ij}) = \beta_0 + \beta_1 \text{age}_{ij} + \beta_2 \text{age}_{ij}^2 + \mathbf{x}_{ij}^T\boldsymbol{\gamma} + u_j,
\qquad u_j \sim \mathcal{N}(0, \tau^2)$$

where $\mathbf{x}_{ij}$ collects tumor size, grade, ER status, lymph nodes and chemotherapy (all standardized where continuous). The quadratic age term captures the U-shaped age effect built into the data-generating process.

### Estimation: Penalized Quasi-Likelihood (PQL)

PQL (Breslow & Clayton, 1993) alternates between:
1. **Fitting a Gaussian linear mixed model** to a *working variable* $z = \eta + (y - \pi)\,\partial\eta/\partial\pi$ with iteratively updated weights $w_i = \pi_i(1-\pi_i)$,
2. **Updating the variance component** $\tau^2$ from the LMM fit,

until convergence. It approximates the marginal likelihood by a Laplace expansion around the random effects.

**Known limitations (they matter for interpretation below):**
- **Attenuation bias**: PQL underestimates variance components and slightly shrinks fixed effects for binary outcomes, especially with small cluster sizes or weak clustering. With 50 patients/hospital and a strong center effect the estimates are usable but biased toward zero.
- The reported log-likelihood/AIC are **conditional quasi-likelihood** quantities — they are *not* comparable to the genuine ML-based AIC of a GLM.
- The current implementation supports a **single random-effect term**.

### Intraclass Correlation (latent scale)

$$\text{ICC} = \frac{\tau^2}{\tau^2 + \pi^2/3}$$

using the logistic latent-variable residual variance $\pi^2/3 \approx 3.29$.

### Interpretation

$e^{\beta_k}$ is the **odds ratio** for a one-unit increase in predictor $k$, *conditional on the hospital* (subject-specific interpretation).


## 5. Model Fitting

Null model (random intercept only) first for the ICC, then the full binomial GAMM via PQL.

In [ ]:
print("="*80)
print("MODEL 1: NULL MODEL (Random Intercept per Hospital, Binomial PQL)")
print("="*80)

start_time = time.time()
result_null = fit_gamm(
    formula='survived ~ 1 + (1 | hospital_id)',
    data=df,
    family='binomial'
)
time_null = time.time() - start_time

print(f"\nConverged: {result_null.converged} (PQL iterations: {result_null.n_iterations})")
print(f"Time: {time_null:.3f}s")

# Variance components and ICC on the latent scale
tau_sq = result_null.variance_components[0][0, 0]
icc = tau_sq / (tau_sq + np.pi**2 / 3)

print(f"\nVariance Components:")
print(f"   Between-hospital (tau^2): {tau_sq:.4f}  (true: 1.00 - PQL attenuation expected)")
print(f"   ICC (latent scale): {icc:.4f} ({icc*100:.1f}%)")
print(f"\nInterpretation: {icc*100:.0f}% of the latent variability in survival")
print(f"is attributable to differences between hospitals.")
print("="*80)


In [ ]:
print("="*80)
print("MODEL 2: FULL MODEL (Binomial GAMM via PQL)")
print("="*80)

start_time = time.time()
result_full = fit_gamm(
    formula='survived ~ age_std + age_std_sq + tumor_std + grade + er_positive + ln_std + chemo + (1 | hospital_id)',
    data=df,
    family='binomial'
)
time_full = time.time() - start_time

print(f"\nConverged: {result_full.converged} (PQL iterations: {result_full.n_iterations})")
print(f"Time: {time_full:.3f}s")

print(f"\nFixed Effects (log-odds and odds ratios):")
pred_names = ['Intercept', 'Age', 'Age^2', 'Tumor', 'Grade', 'ER+', 'LymphNodes', 'Chemo']
print(f"{'Predictor':<15}{'beta':>10}{'OR':>10}")
print("-" * 35)
for name, coef in zip(pred_names, result_full.beta_parametric):
    print(f"{name:<15}{coef:>+10.4f}{np.exp(coef):>10.3f}")

tau_sq_full = result_full.variance_components[0][0, 0]
print(f"\nHospital variance (tau^2): {tau_sq_full:.4f} (true: 1.00)")
print("\nReading: OR > 1 raises the odds of 5-year survival, holding hospital")
print("effects fixed (conditional interpretation). ER+ status and chemotherapy")
print("raise the odds; higher grade, larger tumors and more positive lymph")
print("nodes lower them. The quadratic age term is positive: the age effect is")
print("U-shaped, as built into the data-generating process.")
print("="*80)


## 6. Residual Diagnostics

For a binary outcome, individual residuals are uninformative; we use **binned residuals** (mean response residual within groups of similar fitted probability) and a **calibration plot** (observed survival rate vs mean fitted probability per bin).

In [ ]:
print("="*80)
print("RESIDUAL DIAGNOSTICS (full binomial GAMM)")
print("="*80)

prob_full = np.asarray(result_full.fitted_values)  # probability scale
y_bin = df['survived'].values.astype(float)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Binned residuals vs fitted
bins = pd.qcut(prob_full, q=20, duplicates='drop')
binned = pd.DataFrame({'p': prob_full, 'r': y_bin - prob_full}).groupby(bins, observed=True).agg(
    mean_p=('p', 'mean'), mean_r=('r', 'mean'))
axes[0].plot(binned['mean_p'], binned['mean_r'], 'o-', color='steelblue')
axes[0].axhline(0, color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('Fitted probability (binned)')
axes[0].set_ylabel('Mean residual')
axes[0].set_title('Binned Residuals vs Fitted', fontweight='bold')
axes[0].grid(alpha=0.3)

# Calibration
cal_bins = pd.qcut(prob_full, q=15, duplicates='drop')
cal = pd.DataFrame({'p': prob_full, 'y': y_bin}).groupby(cal_bins, observed=True).agg(
    mean_pred=('p', 'mean'), obs=('y', 'mean'))
axes[1].plot([0, 1], [0, 1], 'r--', linewidth=2, label='Perfect calibration')
axes[1].plot(cal['mean_pred'], cal['obs'], 'b-o', markersize=4)
axes[1].set_xlabel('Mean predicted probability')
axes[1].set_ylabel('Observed survival rate')
axes[1].set_title('Calibration Plot', fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

cal_err = np.abs(cal['obs'] - cal['mean_pred']).mean()
print(f"\nMax |binned residual|:  {np.abs(binned['mean_r']).max():.4f}")
print(f"Mean calibration error: {cal_err:.4f}")
print("   ✓ Well calibrated" if cal_err < 0.05 else "   ⚠ Calibration error above 5 points")
print("="*80)

## 7. Conditional vs Marginal Effects and Hospital Random Effects

In [ ]:
print("="*80)
print("COMPARISON: BINOMIAL GLM (ignoring clustering) vs BINOMIAL GAMM (PQL)")
print("="*80)

# Marginal model: plain logistic GLM, no hospital effect
X_fixed = df[['age_std', 'age_std_sq', 'tumor_std', 'grade', 'er_positive', 'ln_std', 'chemo']].values
result_glm = fit_glm(X=X_fixed, y=df['survived'].values.astype(float), family='binomial')

beta_glm = np.concatenate([[result_glm.intercept_], result_glm.coef_])
beta_gamm = result_full.beta_parametric

print(f"\n{'Predictor':<15}{'GLM (marginal)':>16}{'GAMM (conditional)':>20}")
print("-" * 52)
for name, b_g, b_m in zip(pred_names, beta_glm, beta_gamm):
    print(f"{name:<15}{b_g:>+16.4f}{b_m:>+20.4f}")
print("-" * 52)

print("\nTwo honest observations:")
print("1. GAMM coefficients are slightly LARGER in magnitude: conditional")
print("   (hospital-specific) effects exceed marginal (population-average) effects")
print("   in logistic models - a well-known property of the non-linear logit link.")
print("2. We do NOT compare AICs: the GAMM's quasi-likelihood AIC and the GLM's")
print("   ML-based AIC are on different scales and a comparison would be invalid.")
print("="*80)


In [ ]:
# Hospital random effects visualization
print("="*80)
print("HOSPITAL RANDOM EFFECTS")
print("="*80)

# Extract random effects
re = result_full.random_effects
hospital_re = list(re.values())[0]
re_values = np.array([v[0] for v in hospital_re.values()])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Distribution
axes[0].hist(re_values, bins=15, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('Random Effect')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Hospital Effects', fontweight='bold')

# Caterpillar plot
sorted_idx = np.argsort(re_values)
axes[1].errorbar(range(len(re_values)), re_values[sorted_idx], 
                 fmt='o', markersize=4, capsize=3, alpha=0.7)
axes[1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Hospital (sorted)')
axes[1].set_ylabel('Random Effect')
axes[1].set_title('Caterpillar Plot', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nHospital Effects Range: {re_values.min():.3f} to {re_values.max():.3f}")
print(f"Standard Deviation: {re_values.std():.3f}")
print("="*80)

## 8. Conclusions

### Key Findings
1. **Hospital clustering is real**: the null-model ICC is ~0.11 on the latent scale (τ̂² ≈ 0.41 against a true value of 1.00 — the attenuation is the known PQL bias for binary outcomes, not absence of clustering).
2. **Tumor size, grade and lymph nodes** lower the odds of 5-year survival (OR < 1); **ER+ status and chemotherapy** raise them (OR > 1).
3. **Age effect is U-shaped**: the quadratic term is positive, matching the data-generating process.
4. **Conditional vs marginal**: GAMM coefficients are slightly larger in magnitude than GLM coefficients, as theory predicts for logistic mixed models.

### Limitations

- **PQL attenuation**: variance components (and, to a lesser extent, fixed effects) are biased towards zero for binary outcomes; with 50 patients/hospital the bias is moderate but visible. A Laplace or adaptive quadrature estimator would reduce it.
- **Single random-effect term**: the current implementation cannot add random slopes (e.g., treatment effect varying by hospital).
- **Quasi-likelihood inference**: AIC/log-likelihood from PQL are conditional approximations, not comparable to GLM AICs.
- **Binary endpoint, not survival analysis**: no censoring, no time-to-event, no hazards. A real oncology dataset with follow-up times and censoring requires a survival framework (Cox, Kaplan–Meier, parametric hazards), which Aurora-GLM does not provide.
- The hospital-effect strength in the synthetic data was set to sd = 1.0 on the logit scale precisely so that PQL can recover it; weaker clustering collapses to τ̂² = 0.

### Aurora-GLM Capabilities Demonstrated
- Binomial GAMM via PQL (`fit_gamm(..., family='binomial')`)
- Random intercepts and ICC on the latent scale
- Odds-ratio interpretation of mixed-model coefficients
- Conditional (GAMM) vs marginal (GLM) comparison

---
**Dataset**: Simulated multi-center breast cancer study (N=1,500 patients, 30 hospitals × 50 patients)

### References

- Breslow, N. E., & Clayton, D. G. (1993). Approximate inference in generalized linear mixed models. *Journal of the American Statistical Association*, 88(421), 9-25.
- Molenberghs, G., & Verbeke, G. (2005). *Models for Discrete Longitudinal Data*. Springer. (PQL bias for binary outcomes)
- Stroup, W. W. (2012). *Generalized Linear Mixed Models: Modern Concepts, Methods and Applications*. CRC Press. (conditional vs marginal interpretation)
- Hosmer, D. W., Lemeshow, S., & May, S. (2008). *Applied Survival Analysis* (2nd ed.). Wiley. (for what a true survival analysis would require)
